# TUTORIAL: Class EchoStateNetwork

Introduction to the `tools.esn_core` class. The echo state network class is based on Raca & Racca Magri (2021).

In this tutorial we will use an EchoStateNetwork to model the Lorenz 63 system. 


In [ ]:
from romda.tools import EchoStateNetwork
# help(EchoStateNetwork)

# Create training data from the Lorenz 63 model

In [ ]:
from romda.utils import create_Lorenz63_dataset

# Create a Lorenz63 dataset with noise
dataset = create_Lorenz63_dataset(noise_level=0.02, num_lyap_times = 300)[0]

clean_data = dataset['clean_data']
noisy_data = dataset['noisy_data']
t = dataset['t']
N_lyap = dataset['N_lyap']

### Separate data into train, validation and test datasets and visualize

In [ ]:
N_transient = 15 * N_lyap
N_val = 5 * N_lyap
N_train = 60 * N_lyap - N_val 
N_test = noisy_data.shape[0]-sum((N_train,N_val,N_transient))


train_val_data = noisy_data[N_transient:N_transient+N_train+2*N_val]

test_data = noisy_data[N_transient+N_train+N_val:N_transient+N_train+N_val+N_test]
test_data_clean = clean_data[N_transient+N_train+N_val:N_transient+N_train+N_val+N_test] # Truth to compare the tests


In [ ]:
from romda.plotting import plot_train_dataset

dt = t[1] - t[0]
t_lyap = N_lyap * dt

split_times = [tt / N_lyap for tt in [N_transient, N_train, N_val, N_test]]

plot_train_dataset(clean_data, noisy_data, t/t_lyap, *split_times)

# Initialize the ESN
NB: Different initializations result in different Wout.

In [ ]:
import numpy as np

ESN_params = dict(N_wash=10,  # Number of washout steps i.e., open-loop initialization
                  N_units=50,  # Number of neurons 
                  upsample=2, # We want the ESN to predict t + upsample * dt 
                  t_train=N_train * dt,  # Training time
                  t_val=N_val * dt, # Validation time
                  t_test=2*N_val * dt, # Validation time
                  # Training-specific input_parameters
                  N_func_evals=30,
                  N_grid=5,
                  noise=1e-2,
                  Win_type='sparse',
                  N_folds=8,
                  N_split=5,
                  # Hyperparameter optimization ranges
                  rho_range=(.2, .95),
                  sigma_in_range=(-2, 2),
                  tikh_range=[1E-9, 1E-10],
                #   random_initialization=True
                  )


# Initialize ESN class
ESN_init = EchoStateNetwork(y=np.zeros((3,1)), 
                       dt=dt, 
                       **ESN_params)


# Train the ESN
The result of the ESN training is the output matrix. We can train the network to input the full state or partial observations.

## a) Full observability


In [ ]:
seeds= [0, 252]
for seed in seeds:
    ESN = ESN_init.copy()
    ESN.seed_W = seed
    ESN.train(train_val_data, 
              add_noise=True, 
              plot_training=seed==seeds[-1]
             )

### Test the long-term prediction

## b) Partial observability

In [ ]:
N_transient = 15 * N_lyap
N_val = 5 * N_lyap
N_train = 120 * N_lyap - N_val 
N_test = noisy_data.shape[0]-sum((N_train,N_val,N_transient))

train_val_data = noisy_data[N_transient:N_transient+N_train+2*N_val]


# Define Echo state network input_parameters

ESN_partial = ESN_init.copy()
ESN_partial.N_folds = 12
ESN_partial.rho_range = (0.4, 0.95)

# Set the observed dimensions
ESN_partial.observed_idx = [0, 2]


ESN_partial.train(train_val_data, 
          add_noise=True, 
          plot_training=True,
         )

If prediction is much worse, try increasing the length of the timeseries or number of folds.